# Fraud Detection Model Training and Top-10 Feature Selection

## Objective
Select the strongest fraud drivers, train a model using only the top features, and evaluate performance in a way that supports explainable alert prioritisation.


In [ ]:
# Import the core libraries used for data analysis and visualisation.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make notebook tables easier to inspect during review.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Add the project src folder so notebook code reuses production pipeline logic.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT / 'src'))

from fraud_pipeline import load_transactions, train_top_feature_model, TARGET


In [ ]:
# Load transactions through the reusable project pipeline.
transactions = load_transactions(REPO_ROOT)

print('Transactions shape:', transactions.shape)
print('Fraud rate:', f"{transactions[TARGET].mean():.2%}")


## Top-10 Feature Selection
The reviewer asked for top features using SHAP or mutual information. This notebook uses mutual information because it is direct, fast, and easy to explain: it measures how much each feature reduces uncertainty about the fraud label.


In [ ]:
# Train the final model after selecting the top 10 mutual-information features.
bundle = train_top_feature_model(transactions, top_n=10)

print('Training rows:', bundle.train_rows)
print('Testing rows:', bundle.test_rows)
print('Selected top features:')
for rank, feature in enumerate(bundle.top_features, start=1):
    print(f'{rank}. {feature}')


In [ ]:
# Display the top ranked features and their mutual-information scores.
display(bundle.feature_importance.head(10))


## Model Evaluation
ROC-AUC is used because fraud detection is a ranking problem: analysts need high-risk transactions surfaced before low-risk transactions.


In [ ]:
# Summarise final top-feature model performance.
print('ROC-AUC:', round(bundle.metrics['roc_auc'], 4))
report = pd.DataFrame(bundle.metrics['classification_report']).T
display(report)


## Modelling Insight
The final model intentionally uses only the top 10 selected features. This makes the application easier to explain to analysts and aligns the deployed app with the most important fraud drivers.
